<a href="https://www.kaggle.com/code/jayhawk1900/f1-tabm?scriptVersionId=321464369" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# F1 Pit Stop Prediction — TabM (Parameter-Efficient Neural Ensemble)

**Competition:** [Playground Series S6E5 — Predicting F1 Pit Stops](https://www.kaggle.com/competitions/playground-series-s6e5)
**Metric:** ROC-AUC | **Task:** Binary classification — will this driver pit on the next lap?

---

## Overview

**TabM** is a tabular neural network (Yandex Research, 2024) built around **BatchEnsemble**: instead of training k independent networks, it uses one shared backbone weight matrix plus k lightweight per-member rank-1 adapters (two small vectors each). Each member scales the input and output by its own vectors, so they learn different functions and make different errors — a real ensemble at nearly single-model cost.

This is a deliberately **plain** TabM (no periodic numeric embeddings) to maximize error diversity from the RealMLP model, so the two blend well together. It uses the same feature-engineering pipeline and ORIG-concat strategy as the RealMLP notebook, with multi-seed averaging.

## Step 1: The BatchEnsemble Linear Layer

The single novel component. A standard linear layer has one weight matrix `W`. BatchEnsemble shares `W` across all `k` ensemble members but gives each member two cheap vectors:
- `r_i` (input scaling, length = in_features)
- `s_i` (output scaling, length = out_features)

Member `i` computes: `((x * r_i) @ W) * s_i + bias_i`

This is equivalent to each member having weight `W ⊙ (r_i ⊗ s_i)` — a per-member rank-1 modulation of the shared backbone. The bulk of the parameters (`W`) is shared; only the small `r`, `s`, and bias vary per member. Result: ensemble diversity at a fraction of the cost of `k` full models.

In [1]:
import math, random, time, warnings
import numpy as np, pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:', torch.__version__, '| device:', device)


class BatchEnsembleLinear(nn.Module):
    """Linear layer shared across k members, each modulated by rank-1 adapters.

    For member i:  out = ((x * r_i) @ W) * s_i + bias_i
    - W: shared weight (in_features, out_features)
    - r: per-member input scaling (k, in_features)
    - s: per-member output scaling (k, out_features)
    - bias: per-member bias (k, out_features)
    """
    def __init__(self, in_features, out_features, k, bias=True):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.k = k
        self.weight = nn.Parameter(torch.empty(in_features, out_features))
        self.r = nn.Parameter(torch.ones(k, in_features))
        self.s = nn.Parameter(torch.ones(k, out_features))
        self.bias = nn.Parameter(torch.zeros(k, out_features)) if bias else None
        # Init: shared weight like a normal linear layer; r,s start near 1 with small noise
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        with torch.no_grad():
            self.r += 0.01 * torch.randn_like(self.r)
            self.s += 0.01 * torch.randn_like(self.s)

    def forward(self, x):
        # x: (batch, k, in_features)
        # scale input by r, matmul shared W, scale output by s, add bias
        x = x * self.r[None, :, :]                      # (batch, k, in_features)
        x = torch.einsum('bki,io->bko', x, self.weight) # (batch, k, out_features)
        x = x * self.s[None, :, :]
        if self.bias is not None:
            x = x + self.bias[None, :, :]
        return x


# Sanity test
_be = BatchEnsembleLinear(in_features=10, out_features=7, k=4).to(device)
_x = torch.randn(8, 4, 10).to(device)   # (batch=8, k=4 members, in=10)
_out = _be(_x)
print('BatchEnsembleLinear input :', tuple(_x.shape))
print('BatchEnsembleLinear output:', tuple(_out.shape), '= (batch, k, out_features)')

# Verify members actually differ (different r,s → different outputs for same input row)
_same_input = torch.randn(1, 4, 10).to(device)
_o = _be(_same_input).detach().cpu().numpy()[0]  # (k=4, 7)
diffs = np.std(_o, axis=0).mean()
print(f'Mean std across the 4 members (should be > 0, i.e. they differ): {diffs:.4f}')

# Param efficiency check
shared = _be.weight.numel()
per_member = _be.r.numel() + _be.s.numel() + _be.bias.numel()
print(f'Shared W params: {shared} | per-member adapter params: {per_member} (cheap)')
del _be, _x, _out, _same_input

PyTorch: 2.10.0+cu128 | device: cuda
BatchEnsembleLinear input : (8, 4, 10)
BatchEnsembleLinear output: (8, 4, 7) = (batch, k, out_features)
Mean std across the 4 members (should be > 0, i.e. they differ): 0.5700
Shared W params: 70 | per-member adapter params: 96 (cheap)
